# Model Research Workflow (RF baseline + one new model)

This notebook runs the required research using functions from `code/models/model_research.py`.

- Baseline: Random Forest (`rf_full`)
- New model: Histogram Gradient Boosting (`hgb_base`)
- Includes: grouped CV, temporal holdout ROC/metrics, diagnostics, post-hoc error EDA, stability analysis, and unlabeled-image sanity checks.

In [4]:
from pathlib import Path
import sys
import joblib
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

# Make repo importable when running from this notebook folder
repo_root = Path.cwd().resolve().parents[1]
code_dir = repo_root / 'code'
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

from models import model_research as mr

print('Repo root:', repo_root)
print('Results dir:', mr.RESULTS_DIR)
print('RF pkl path:', repo_root / 'results' / 'best_rf_model.pkl')

Repo root: /Users/zhangshizhe/Dropbox/UCB/Course/214_STAT/self/lab2
Results dir: /Users/zhangshizhe/Dropbox/UCB/Course/214_STAT/self/lab2/report/model_research
RF pkl path: /Users/zhangshizhe/Dropbox/UCB/Course/214_STAT/self/lab2/results/best_rf_model.pkl


In [5]:
# 1) Load data, model definitions, and pretrained RF

df_train, df_test, df_all = mr.load_labeled_data()
specs = mr.model_specs()

rf_pkl_path = repo_root / 'results' / 'best_rf_model.pkl'
rf_model = joblib.load(rf_pkl_path)

print('Train shape:', df_train.shape)
print('Test shape :', df_test.shape)
print('All shape  :', df_all.shape)
print('Models     :', list(specs.keys()))
print('Loaded RF  :', rf_model.__class__.__name__)

/opt/homebrew/Caskroom/miniconda/base/envs/env_214/lib/python3.14/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Train shape: (125598, 45)
Test shape : (82083, 45)
All shape  : (207681, 45)
Models     : ['rf_full', 'hgb_base']
Loaded RF  : RandomForestClassifier


/opt/homebrew/Caskroom/miniconda/base/envs/env_214/lib/python3.14/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
# 2) Grouped CV for the NEW model only (HGB)

specs_hgb = {'hgb_base': specs['hgb_base']}
cv_df = mr.run_logo_cv(df_all, specs_hgb)
cv_df[cv_df['fold'] == 'mean'].sort_values('cv_roc_auc', ascending=False)

,model,fold,held_out_image,cv_roc_auc
3,hgb_base,mean,all,0.952341


In [8]:
# 3) Temporal holdout evaluation
#    - HGB: trained in this notebook
#    - RF : loaded from results/best_rf_model.pkl

# HGB evaluation from helper
metrics_hgb, fitted_models = mr.evaluate_temporal_holdout(df_train, df_test, specs_hgb)

# RF evaluation from pretrained pkl
rf_features = list(getattr(rf_model, 'feature_names_in_', mr.FULL_FEATURES))
X_test_rf = df_test[rf_features].values
y_test = df_test['label'].values

y_pred_rf = rf_model.predict(X_test_rf)
y_prob_rf = rf_model.predict_proba(X_test_rf)[:, 1]

rf_metrics_row = {
    'model': 'rf_full',
    'accuracy': accuracy_score(y_test, y_pred_rf),
    'f1': f1_score(y_test, y_pred_rf, pos_label=1),
    'precision': precision_score(y_test, y_pred_rf, pos_label=1),
    'recall': recall_score(y_test, y_pred_rf, pos_label=1),
    'roc_auc': roc_auc_score(y_test, y_prob_rf),
    'n_features': len(rf_features),
    'assumptions': 'Pretrained RandomForest loaded from results/best_rf_model.pkl'
}

fitted_models['rf_full'] = {'model': rf_model, 'y_pred': y_pred_rf, 'y_prob': y_prob_rf}
metrics_df = (
    pd.concat([metrics_hgb, pd.DataFrame([rf_metrics_row])], ignore_index=True)
    .sort_values('roc_auc', ascending=False)
    .reset_index(drop=True)
)
metrics_df

/opt/homebrew/Caskroom/miniconda/base/envs/env_214/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/env_214/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


,accuracy,f1,precision,recall,roc_auc,model,n_features,assumptions
0,0.960832,0.960467,0.928290,0.994956,0.995764,hgb_base,8,Additive boosted trees; no normality/linearity...
1,0.955789,0.955412,0.922726,0.990498,0.987843,rf_full,41,Pretrained RandomForest loaded from results/be...


In [9]:
# 4) Assumption reasonableness: temporal distribution shift checks
shift_checks = mr.check_distribution_shift(df_train, df_test)
shift_checks.sort_values('abs_smd_test_vs_train', ascending=False)

,feature,train_skew,test_skew,abs_smd_test_vs_train
7,NDAI_DF_AF,2.481194,1.226006,1.146794
2,DF,-0.676149,-1.761528,0.747744
1,CORR,-0.639272,0.098574,0.537286
3,CF,-0.916312,-2.089033,0.398297
6,AN,-1.632256,-1.253636,0.229945
5,AF,-1.391681,-1.275895,0.202640
0,SD,3.236782,1.823850,0.081067
4,BF,-1.116833,-1.499847,0.038084


In [10]:
# 5) Deep diagnostics for the new model only
selected_name = 'hgb_base'
mr.model_diagnostics(selected_name, fitted_models[selected_name], df_test, specs)
mr.posthoc_error_eda(selected_name, fitted_models[selected_name], df_test)
print('Diagnostics done for', selected_name)

Diagnostics done for hgb_base


In [11]:
# 6) Stability analysis + sanity check on unlabeled images
mr.stability_analysis(selected_name, specs, df_train, df_test)
unlabeled_df = mr.predict_on_unlabeled_images(specs, df_train, model_name=selected_name)
unlabeled_df

ValueError: Shape of passed values is (115378, 10), indices imply (115378, 11)

In [12]:
# 7) Save report-ready figures + generate model-section outline
from pathlib import Path
import shutil
import numpy as np

report_dir = repo_root / 'report'
model_results_dir = report_dir / 'model_research'
fig_dir = report_dir / 'figures' / 'model'
fig_dir.mkdir(parents=True, exist_ok=True)

# Copy/rename key figures for the final report
figure_map = {
    model_results_dir / 'temporal_holdout_roc.png': fig_dir / 'fig_model_roc_temporal_holdout.png',
    model_results_dir / 'hgb_base_confusion_matrix.png': fig_dir / 'fig_model_hgb_confusion_matrix.png',
    model_results_dir / 'hgb_base_perm_importance_top15.png': fig_dir / 'fig_model_hgb_permutation_importance.png',
    model_results_dir / 'hgb_base_convergence.png': fig_dir / 'fig_model_hgb_convergence.png',
    model_results_dir / 'hgb_base_spatial_error_map.png': fig_dir / 'fig_model_hgb_spatial_errors.png',
}

copied = []
missing = []
for src, dst in figure_map.items():
    if src.exists():
        shutil.copy2(src, dst)
        copied.append(dst.name)
    else:
        missing.append(src.name)

# Build model-section outline from current notebook results
metrics_sorted = metrics_df.sort_values('roc_auc', ascending=False).reset_index(drop=True)
best_row = metrics_sorted.iloc[0]
rf_row = metrics_sorted[metrics_sorted['model'] == 'rf_full'].iloc[0]
hgb_row = metrics_sorted[metrics_sorted['model'] == 'hgb_base'].iloc[0]

cv_mean_hgb = (
    cv_df[(cv_df['model'] == 'hgb_base') & (cv_df['fold'] == 'mean')]['cv_roc_auc'].iloc[0]
    if ((cv_df['model'] == 'hgb_base') & (cv_df['fold'] == 'mean')).any()
    else np.nan
)

top_shift = shift_checks.sort_values('abs_smd_test_vs_train', ascending=False).head(3)
shift_lines = [
    f"- {row.feature}: |SMD|={row.abs_smd_test_vs_train:.3f}, train skew={row.train_skew:.3f}, test skew={row.test_skew:.3f}"
    for row in top_shift.itertuples(index=False)
]

outline_lines = [
    '# Model Section Outline (Lab 2 Report)',
    '',
    '## 1) Problem Setup',
    '- Target: cloud presence classification (+1 cloud, -1 not cloud).',
    '- Data split: temporal holdout (train on O012791 + O013257, test on O013490).',
    '- Features: engineered MISR features; RF uses full engineered + AE features from saved model.',
    '',
    '## 2) Models Compared',
    '- Baseline: Random Forest (loaded from `results/best_rf_model.pkl`).',
    '- New model: Histogram Gradient Boosting (HGB) trained in notebook.',
    '',
    '## 3) Model Assumptions and Reasonableness',
    '- Tree ensembles are nonparametric and do not require linearity/normality.',
    '- Temporal stability assessed with train-vs-test distribution shift (SMD/skew).',
    '- Largest shifts observed:',
    *shift_lines,
    '',
    '## 4) Fit Assessment',
    f"- HGB grouped CV (leave-one-image-out) mean ROC AUC: {cv_mean_hgb:.4f}.",
    f"- Temporal holdout RF: AUC={rf_row.roc_auc:.4f}, F1={rf_row.f1:.4f}, Accuracy={rf_row.accuracy:.4f}.",
    f"- Temporal holdout HGB: AUC={hgb_row.roc_auc:.4f}, F1={hgb_row.f1:.4f}, Accuracy={hgb_row.accuracy:.4f}.",
    f"- Best temporal model in current notebook run: {best_row.model} (AUC={best_row.roc_auc:.4f}).",
    '',
    '## 5) Selected Model Diagnostics (HGB)',
    '- ROC curve (temporal holdout).',
    '- Confusion matrix.',
    '- Permutation feature importance (top 15).',
    '- Convergence curve (train/validation loss vs boosting iteration).',
    '',
    '## 6) Post-hoc Error EDA (HGB)',
    '- Spatial map of false positives/false negatives.',
    '- Error-rate by feature quantiles (e.g., SD, CORR, NDAI_DF_AF).',
    '- Discuss where and why errors cluster.',
    '',
    '## 7) Stability and Future-Data Readiness',
    '- Seed perturbation stability results.',
    '- Input-noise perturbation stability results.',
    '- Comment on robustness and expected performance drift on future unlabeled orbits.',
    '',
    '## 8) Figures to Include',
    '- fig_model_roc_temporal_holdout.png',
    '- fig_model_hgb_confusion_matrix.png',
    '- fig_model_hgb_permutation_importance.png',
    '- fig_model_hgb_convergence.png',
    '- fig_model_hgb_spatial_errors.png',
]

outline_path = report_dir / 'model_section_outline.md'
outline_path.write_text('\n'.join(outline_lines), encoding='utf-8')

print(f'Copied {len(copied)} figure(s) to: {fig_dir}')
if copied:
    print('  - ' + '\n  - '.join(copied))
if missing:
    print('Missing source figure(s):')
    print('  - ' + '\n  - '.join(missing))
print(f'Outline generated: {outline_path}')

Copied 5 figure(s) to: /Users/zhangshizhe/Dropbox/UCB/Course/214_STAT/self/lab2/report/figures/model
  - fig_model_roc_temporal_holdout.png
  - fig_model_hgb_confusion_matrix.png
  - fig_model_hgb_permutation_importance.png
  - fig_model_hgb_convergence.png
  - fig_model_hgb_spatial_errors.png
Outline generated: /Users/zhangshizhe/Dropbox/UCB/Course/214_STAT/self/lab2/report/model_section_outline.md


## Outputs

All generated tables/plots are saved in `report/model_research/`, including:
- `logo_cv_auc.csv` (HGB grouped CV)
- `temporal_test_metrics.csv` and `temporal_holdout_roc.png`
- `distribution_shift_checks.csv`
- `hgb_base_*` diagnostics and post-hoc EDA files
- `unlabeled_sanity_hgb_base.csv` and probability maps
- `summary.md`

Random Forest in this notebook is loaded from `results/best_rf_model.pkl` (no RF retraining).